# Country feature enrichment

In this notebook, we enrich the countries from the matched Sinas data with features that may inform the horizon scanning GNN.

## 1. Load in dependencies and data

In [10]:
import os
import pandas as pd
from pathlib import Path
import requests
import numpy as np
import time
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import torch
import country_converter as coco
from collections import defaultdict
from tqdm.auto import tqdm
import logging

In [2]:
repo_root         = Path.cwd().parent # get repo root
sinas_path        = repo_root / 'data' / 'sinas_matched_species.csv' #get df file path

df_sinas = pd.read_csv(sinas_path) #read in df
df_sinas.head() #check structure


,location,locationID,taxon,taxonID,eventDate,habitat,occurrenceStatus,establishmentMeans,degreeOfEstablishment,pathway,...,gbif_id,gbif_genus_id,gbif_family_id,gbif_order_id,gbif_class_id,gbif_phylum_id,gbif_kingdom_id,gbif_match_type,gbif_confidence,gbif_rank
0,Aegean,101,Camponotus fallax,5586,NaN,terrestrial,NaN,native,NaN,NaN,...,1312649.0,1312361.0,4342.0,1457.0,216.0,54.0,1.0,EXACT,99.0,SPECIES
1,Aegean,101,Camponotus vagus,5601,NaN,terrestrial,NaN,native,NaN,NaN,...,1313740.0,1312361.0,4342.0,1457.0,216.0,54.0,1.0,EXACT,99.0,SPECIES
2,Aegean,101,Cardiocondyla mauritanica,5609,NaN,terrestrial,NaN,introduced,NaN,NaN,...,1317123.0,1317100.0,4342.0,1457.0,216.0,54.0,1.0,EXACT,99.0,SPECIES
3,Aegean,101,Cataglyphis nodus,5620,NaN,NaN,NaN,native,NaN,NaN,...,1319599.0,1319559.0,4342.0,1457.0,216.0,54.0,1.0,EXACT,99.0,SPECIES
4,Aegean,101,Crematogaster scutellaris,5716,NaN,terrestrial,NaN,native,NaN,NaN,...,1325120.0,1324306.0,4342.0,1457.0,216.0,54.0,1.0,EXACT,99.0,SPECIES


## 2. Extract country features and save as new file for enrichment

In [3]:
#extract unique locations
unique_locations = df_sinas['location'].dropna().unique()

#sort locations alphabetically
all_country_names = sorted(list(unique_locations))

# print summary diagnostics
print(f"Total unique regions/countries found: {len(all_country_names)}")
print("First 10 entries as a sample:", all_country_names[:10])

# export to a clean reference file for your feature collection step
with open('sinas_country_names.txt', 'w', encoding='utf-8') as f:
    for country in all_country_names:
        f.write(f"{country}\n")

Total unique regions/countries found: 289
First 10 entries as a sample: ['Aegean', 'Afghanistan', 'Akrotiri and Dhekelia', 'Alaska', 'Albania', 'Algeria', 'American Samoa', 'Amsterdam Island', 'Andorra', 'Angola']


## 3. Enrich regions with climate data

In [20]:
# --- CONFIGURATION ---
MASTER_TXT = "gnn_country_names.txt"
COORD_CSV = "country_coordinates.csv"
OUTPUT_CSV = "aligned_country_climate.csv"
START_DATE = "2015-01-01"
END_DATE = "2024-12-31"
# ---------------------

print("🏁 Initializing Climate Enrichment Pipeline...")

# === STEP 1: LOAD MASTER TARGET LIST ===
if not os.path.exists(MASTER_TXT):
    raise FileNotFoundError(f"❌ Could not find master file '{MASTER_TXT}'. Please generate it first.")

with open(MASTER_TXT, "r", encoding="utf-8") as f:
    master_locations = [line.strip() for line in f if line.strip()]

print(f"📋 Loaded {len(master_locations)} target locations from graph registry.")


# === STEP 2: CHECK EXISTING PROGRESS ===
if os.path.exists(OUTPUT_CSV) and os.path.getsize(OUTPUT_CSV) > 0:
    df_existing = pd.read_csv(OUTPUT_CSV)
    # A country is fully processed if its core features are completely filled out
    successful_locations = set(df_existing[df_existing["mean_temperature"].notna()]["location"])
    df_existing = df_existing[df_existing["location"].isin(successful_locations)]
else:
    df_existing = pd.DataFrame()
    successful_locations = set()

print(f"📊 Progress Check: {len(successful_locations)} / {len(master_locations)} countries already completed.")


# === STEP 3: RESOLVE & CACHE COORDINATES ===
known_coords = {}

# Strategy A: Harvest coordinates from your successfully completed climate rows
if not df_existing.empty:
    df_valid_geo = df_existing[df_existing["latitude"].notna() & df_existing["longitude"].notna()]
    for _, row in df_valid_geo.iterrows():
        known_coords[row["location"]] = (row["latitude"], row["longitude"])

# Strategy B: Load from existing coordinate cache if present
if os.path.exists(COORD_CSV):
    df_cache = pd.read_csv(COORD_CSV)
    for _, row in df_cache.iterrows():
        known_coords[row["location"]] = (row["latitude"], row["longitude"])

# Strategy C: Call geocoding API ONLY for remaining missing targets
missing_coords = [loc for loc in master_locations if loc not in known_coords]

if missing_coords:
    print(f"🌍 Geocoding {len(missing_coords)} missing region coordinates...")
    geolocator = Nominatim(user_agent="gnn_climate_pipeline_v4")
    geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.2) # Enforce OSM terms of service

    for loc in tqdm(missing_coords, desc="Geocoding Progress"):
        try:
            geo = geocode(loc)
            if geo:
                known_coords[loc] = (geo.latitude, geo.longitude)
        except Exception:
            time.sleep(2.0)  # Breather delay if network stutters
            continue

    # FIXED: Indexing into the tuple 'v' split out values to separate lat/lon columns
    coord_rows = [{"location": k, "latitude": v[0], "longitude": v[1]} for k, v in known_coords.items()]
    df_coords = pd.DataFrame(coord_rows)
    df_coords.to_csv(COORD_CSV, index=False)
else:
    # FIXED: Indexing into the tuple 'v' split out values to separate lat/lon columns
    coord_rows = [{"location": k, "latitude": v[0], "longitude": v[1]} for k, v in known_coords.items()]
    df_coords = pd.DataFrame(coord_rows)

print("✅ Coordinate map resolution complete.")


# === STEP 4: ISOLATE WORK BACKLOG ===
# Filter out the 92 countries you already have so we ONLY request missing data
df_todo_coords = df_coords[~df_coords["location"].isin(successful_locations)].reset_index(drop=True)
print(f"📡 Backlog isolated: {len(df_todo_coords)} countries scheduled for climate calculations.\n")


# === STEP 5: ONE-BY-ONE CLIMATE EXTRACTION WITH RETRIES ===
new_climate_records = []

if not df_todo_coords.empty:
    print(f"🚀 Querying Open-Meteo one country at a time...")
    time.sleep(5)  # Initial cooldown after geocoding burst

    for idx, row in tqdm(df_todo_coords.iterrows(), total=len(df_todo_coords), desc="Climate Download Progress"):
        loc_name = row["location"]
        lat = row["latitude"]
        lon = row["longitude"]

        url = "https://archive-api.open-meteo.com/v1/archive"
        params = {
            "latitude": lat,
            "longitude": lon,
            "start_date": START_DATE,
            "end_date": END_DATE,
            "daily": [
                "temperature_2m_mean",
                "temperature_2m_min",
                "precipitation_sum",
                "et0_fao_evapotranspiration",
            ],
            "timezone": "auto",
        }

        backoff = 5  # Start lower; only escalates if this specific country gets rate-limited
        max_backoff = 120  # Cap at 2 minutes so a bad country doesn't stall for 10+ min
        response = None
        while True:
            try:
                response = requests.get(url, params=params, timeout=30)

                if response.status_code == 429:
                    wait = min(backoff, max_backoff)
                    tqdm.write(f"  ⚠️ Rate limited (429) on '{loc_name}'. Cooling down for {wait}s...")
                    time.sleep(wait)
                    backoff *= 2
                    continue
                elif response.status_code != 200:
                    tqdm.write(f"  ❌ API Server Error {response.status_code} for '{loc_name}'. Skipping.")
                    break
                else:
                    break
            except (requests.exceptions.Timeout, requests.exceptions.ConnectionError):
                tqdm.write(f"  ⏳ Connection timed out on '{loc_name}'. Pausing 5s...")
                time.sleep(5.0)
                continue
            except Exception as e:
                tqdm.write(f"  ❌ Unexpected error: {e}. Retrying...")
                time.sleep(5.0)
                continue

        if response is None or response.status_code != 200:
            continue

        # Open-Meteo single-location queries return a flat dictionary payload
        raw_data = response.json()
        daily_payload = raw_data.get("daily", {})
        if not daily_payload or "time" not in daily_payload:
            continue

        df_daily = pd.DataFrame({
            "date": pd.to_datetime(daily_payload["time"]),
            "t_mean": daily_payload["temperature_2m_mean"],
            "t_min": daily_payload["temperature_2m_min"],
            "precip": daily_payload["precipitation_sum"],
            "et0": daily_payload["et0_fao_evapotranspiration"],
        }).dropna()

        if df_daily.empty:
            continue

        # Core Ecogeographical aggregation math
        years_elapsed = len(df_daily) / 365.25
        mean_temp = df_daily["t_mean"].mean()
        monthly_means = df_daily.groupby(df_daily["date"].dt.to_period("M"))["t_mean"].mean()
        temp_seasonality = monthly_means.std()
        annual_frost_days = (df_daily["t_min"] < 0).sum() / years_elapsed
        annual_precipitation = df_daily["precip"].sum() / years_elapsed
        annual_et0 = df_daily["et0"].sum() / years_elapsed
        aridity_index = annual_precipitation / annual_et0 if annual_et0 > 0 else 0

        new_climate_records.append({
            "location": loc_name,
            "latitude": lat,
            "longitude": lon,
            "mean_temperature": round(mean_temp, 2),
            "temperature_seasonality": round(temp_seasonality, 3),
            "frost_days": round(annual_frost_days, 1),
            "precipitation": round(annual_precipitation, 1),
            "aridity_index": round(aridity_index, 3),
        })
        
        # === CHECKPOINT SAVE every 5 countries ===
        if len(new_climate_records) % 5 == 0:
            df_checkpoint = pd.concat([df_existing, pd.DataFrame(new_climate_records)], ignore_index=True)
            df_checkpoint = df_checkpoint.sort_values(by="location").reset_index(drop=True)
            df_checkpoint.to_csv(OUTPUT_CSV, index=False)
            tqdm.write(f"  💾 Checkpoint saved: {len(df_checkpoint)} total records written to '{OUTPUT_CSV}'")

        # Short baseline breather between individual hits to avoid triggering security limits
        time.sleep(1.5)


# === STEP 6: CONSOLIDATE & EXPORT TO GRAPH MATRICES ===
if new_climate_records:
    df_new = pd.DataFrame(new_climate_records)
    # Merge original 92 pristine entries with your freshly retrieved records
    df_final = pd.concat([df_existing, df_new], ignore_index=True)
else:
    df_final = df_existing

# CRITICAL FOR GNN ALIGNMENT: Sort alphabetically by location name
df_final = df_final.sort_values(by="location").reset_index(drop=True)
df_final.to_csv(OUTPUT_CSV, index=False)

print(f"\n🎉 Verification successful! Total records finalized inside '{OUTPUT_CSV}': {len(df_final)}")

🏁 Initializing Climate Enrichment Pipeline...
📋 Loaded 289 target locations from graph registry.
📊 Progress Check: 246 / 289 countries already completed.
✅ Coordinate map resolution complete.
📡 Backlog isolated: 43 countries scheduled for climate calculations.

🚀 Querying Open-Meteo one country at a time...


Climate Download Progress:   2%|▏         | 1/43 [00:49<12:59, 18.57s/it]

  ⏳ Connection timed out on 'Svalbard and Jan Mayen'. Pausing 5s...


Climate Download Progress:   9%|▉         | 4/43 [01:34<10:46, 16.57s/it]

  💾 Checkpoint saved: 251 total records written to 'aligned_country_climate.csv'


...

  ⏳ Connection timed out on 'Zambia'. Pausing 5s...


Climate Download Progress:  98%|█████████▊| 42/43 [24:32<02:19, 139.67s/it]

  ⏳ Connection timed out on 'Zimbabwe'. Pausing 5s...


Climate Download Progress: 100%|██████████| 43/43 [24:46<00:00, 34.56s/it] 


🎉 Verification successful! Total records finalized inside 'aligned_country_climate.csv': 289


In [18]:
#on stalling rate limits

df_emergency = pd.DataFrame(new_climate_records)
df_emergency.to_csv("emergency_backup.csv", index=False)
print(f"Saved {len(df_emergency)} records!")


Saved 44 records!


In [19]:
#merge with existing file

# --- CONFIGURATION ---
OUTPUT_CSV = "aligned_country_climate.csv"
EMERGENCY_CSV = "emergency_backup.csv"  # Change this if you named it differently
# ---------------------

print("🔄 Starting data consolidation...")

# 1. Load the original baseline data (the 92 completed countries)
if os.path.exists(OUTPUT_CSV) and os.path.getsize(OUTPUT_CSV) > 0:
    df_existing = pd.read_csv(OUTPUT_CSV)
    print(f"📋 Loaded {len(df_existing)} base records from '{OUTPUT_CSV}'.")
else:
    df_existing = pd.DataFrame()
    print("⚠️ Base file not found or empty. Starting fresh.")

# 2. Load your 36 newly rescued records
# NOTE: If you didn't save a CSV and the list is still in your notebook RAM,
# use: df_new = pd.DataFrame(new_climate_records)
if os.path.exists(EMERGENCY_CSV):
    df_new = pd.read_csv(EMERGENCY_CSV)
    print(f"📋 Loaded {len(df_new)} rescued records from '{EMERGENCY_CSV}'.")
else:
    raise FileNotFoundError(
        f"❌ Could not find '{EMERGENCY_CSV}'. Make sure the file path matches."
    )

# 3. Concatenate the dataframes
df_final = pd.concat([df_existing, df_new], ignore_index=True)

# 4. Clean up overlaps & strictly align for GNN architecture
# In case a country was partially written or duplicated, keep the freshest data
df_final = df_final.drop_duplicates(subset=["location"], keep="last")

# Sort alphabetically so node indexing matches your graph structure perfectly
df_final = df_final.sort_values(by="location").reset_index(drop=True)

# 5. Overwrite the master file
df_final.to_csv(OUTPUT_CSV, index=False)

print(f"\n🎉 Success! '{OUTPUT_CSV}' has been updated.")
print(f"📊 Total aligned records now available: {len(df_final)}")

🔄 Starting data consolidation...
📋 Loaded 242 base records from 'aligned_country_climate.csv'.
📋 Loaded 44 rescued records from 'emergency_backup.csv'.

🎉 Success! 'aligned_country_climate.csv' has been updated.
📊 Total aligned records now available: 246


# 5. Enrich country dataset with trade features

Important is to first download the BACI trade data master file here (2024 data was used in this case): https://www.cepii.fr/CEPII/en/bdd_modele/bdd_modele_item.asp?id=37

In [8]:
# ==============================================================================
# ECOLOGICAL TRADE PATHWAY PRE-PROCESSING SCRIPT (BACI -> PYTORCH TENSOR)
# ==============================================================================

repo_root         = Path.cwd().parent # get repo root
sinas_path        = repo_root / 'data' / 'sinas_matched_species.csv' #get df file path
baci_path         = repo_root / 'data' / 'BACI_HS22_Y2024_V202601.csv'
output_path       = repo_root / 'data' / 'trade_channels.pt'

print("📖 Reading baseline SInAS geographies to align tensor dimensions...")
df_sinas = pd.read_csv(sinas_path) #read in df
country_col = 'location'

# Extract the exact unique country list and create index mappings
all_country_names = sorted(df_sinas[country_col].dropna().unique().tolist())
co_to_idx = {name: idx for idx, name in enumerate(all_country_names)}
N = len(all_country_names)
print(f"   Found {N} target geographic locations/sub-regions.")

📖 Reading baseline SInAS geographies to align tensor dimensions...
   Found 289 target geographic locations/sub-regions.


In [ ]:
# ==============================================================================
# SILENCE COCO LOGGING SPAM
# ==============================================================================
# country_converter uses standard logging; setting it to ERROR mutes warnings
logging.getLogger('country_converter').setLevel(logging.ERROR)

# ==============================================================================
# SUB-NATIONAL TO SOVEREIGN ISO MAPPING (UPDATED WITH MISSING ISLANDS)
# ==============================================================================
SUB_NATIONAL_TO_ISO = {
    'Aegean': 'GRC', 'Caspian Sea': 'RUS', 'Alaska': 'USA', 'Hawaii': 'USA', 
    'Virgin Islands (U.S.)': 'USA', 'United States Minor Outlying Islands': 'USA', 
    'Puerto Rico': 'USA', 'Guam': 'USA', 'American Samoa': 'USA', 'Northern Mariana Islands': 'USA',
    'Canary Islands': 'ESP', 'Balearic Islands': 'ESP', 'Azores': 'PRT', 'Madeira': 'PRT',
    'Sicily': 'ITA', 'Sardinia': 'ITA', 'Tasmania': 'AUS', 'Lord Howe Islands': 'AUS', 
    'Norfolk Island': 'AUS', 'Christmas Island': 'AUS', 'Cocos Islands': 'AUS',
    'Galapagos': 'ECU', 'Corsica': 'FRA', 'Réunion': 'FRA', 'Mayotte': 'FRA',
    'Guadeloupe': 'FRA', 'Martinique': 'FRA', 'French Guiana': 'FRA',
    'Saint Pierre and Miquelon': 'FRA', 'Saint-Barthélemy': 'FRA', 'Saint-Martin': 'FRA',
    'Amsterdam Island': 'FRA', 'Crozet Islands': 'FRA', 'Kerguelen Islands': 'FRA', 
    'St. Paul Island': 'FRA', 'Scattered Islands': 'FRA', 'Shetland Islands': 'GBR', 
    'Guernsey': 'GBR', 'Jersey': 'GBR', 'Isle of Man': 'GBR', 'Akrotiri and Dhekelia': 'GBR', 
    'Gibraltar': 'GBR', 'Bermuda': 'GBR', 'Anguilla': 'GBR', 'Cayman Islands': 'GBR', 
    'Montserrat': 'GBR', 'Turks and Caicos Islands': 'GBR', 'Virgin Islands (British)': 'GBR', 
    'Falkland Islands': 'GBR', 'Pitcairn Islands': 'GBR', 'Saint Helena': 'GBR', 
    'Ascension': 'GBR', 'Tristan da Cunha': 'GBR', 'Chagos Archipelago': 'GBR', 
    'South Georgia and the South Sandwich Islands': 'GBR', 'Vancouver Island': 'CAN', 
    'Crete': 'GRC', 'Zanzibar Island': 'TZA', 'Rapa Nui': 'CHL', 'Socotra Island': 'YEM', 
    'Nicobar and Andaman Islands': 'IND', 'Rodriguez Island': 'MUS', 'Hong Kong': 'CHN', 
    'Macao': 'CHN', 'Bonaire': 'NLD', 'Saba': 'NLD', 'Sint Eustatius': 'NLD', 
    'Curaçao': 'NLD', 'Sint Maarten': 'NLD', 'Faroe Islands': 'DNK', 'Greenland': 'DNK', 
    'Svalbard and Jan Mayen': 'NOR', 'Bouvet Island': 'NOR', 'Åland': 'FIN', 
    'Tokelau': 'NZL', 'Niue': 'NZL', 'Cook Islands': 'NZL', 'Kermadec Islands': 'NZL', 
    'Northern Cyprus': 'CYP', 'Western Sahara': 'MAR', 'Fernando de Noronha': 'BRA', 
    'Paracel Islands': 'CHN', 'Spratly Islands': 'CHN', 'Palestine': 'PSE', 'Kosovo': 'XKX',
    # Added missing islands from your error logs:
    'Antipodes Island': 'NZL', 
    'Izu Islands': 'JPN', 
    'Ogasawara Islands': 'JPN'
}

cc = coco.CountryConverter()

# Build Name -> ISO3 string map for broadcasting
name_to_iso3 = {}
for name in all_country_names:
    if name in SUB_NATIONAL_TO_ISO:
        name_to_iso3[name] = SUB_NATIONAL_TO_ISO[name]
    else:
        iso3 = cc.convert(names=name, to='ISO3')
        name_to_iso3[name] = 'WLD' if iso3 == 'not_found' else iso3

# Invert dictionary to associate ISO3 strings to matrix index locations
iso3_to_graph_indices = defaultdict(list)
for name, iso3 in name_to_iso3.items():
    iso3_to_graph_indices[iso3].append(co_to_idx[name])

# ==============================================================================
# OPTIMIZED BACI UN CODE MAPPING
# ==============================================================================
print("⚡ Generating numeric-to-ISO3 optimization mappings...")
baci_num_to_iso3 = {}

# Optimization: Instead of checking 1-1000 blindly, extract actual codes present in BACI dataset
if baci_path.exists():
    # Only load the ID columns to save memory and time
    baci_ids = pd.read_csv(baci_path, usecols=['i', 'j'])
    unique_un_codes = list(set(baci_ids['i'].unique()).union(set(baci_ids['j'].unique())))
    
    converted_iso3s = cc.convert(names=unique_un_codes, src='UNcode', to='ISO3')
    
    # Handle single item returns edge-case from coco
    if not isinstance(converted_iso3s, list):
        converted_iso3s = [converted_iso3s]
        
    for num, iso3 in zip(unique_un_codes, converted_iso3s):
        if iso3 != 'not_found':
            baci_num_to_iso3[num] = iso3
else:
    # Fallback to range array if file isn't present yet, relying entirely on the muted logger
    all_possible_un_codes = list(range(1, 1000))
    converted_iso3s = cc.convert(names=all_possible_un_codes, src='UNcode', to='ISO3')
    for num, iso3 in zip(all_possible_un_codes, converted_iso3s):
        if iso3 != 'not_found':
            baci_num_to_iso3[num] = iso3

# Establish multi-channel ecological pathways
ECOLOGICAL_TRADE_MAPPING = {
    'flora_and_timber': ('06', '44'),
    'aquatic_fauna':    ('03',),
    'terrestrial_pets': ('01',),
    'agri_hitchhikers': ('07', '08', '10', '12'),
    'microbial_risk':   ('3002', '3821')
}

channel_keys = list(ECOLOGICAL_TRADE_MAPPING.keys())
num_channels = len(channel_keys)
channel_idx_map = {k: idx for idx, k in enumerate(channel_keys)}

# Allocate multi-channel 3D array: (Channels, Exporters, Importers)
raw_trade_matrices = np.zeros((num_channels, N, N))
print("✅ Array successfully initialized.")

print(raw_trade_matrices.shape)

⚡ Generating numeric-to-ISO3 optimization mappings...
✅ Array successfully initialized without logs flood.
(5, 289, 289)


In [13]:
# ==============================================================================
# CHUNKED PROCESSING OF BACI CSV FILE
# ==============================================================================
print(f"📦 Compiling multi-channel trade profiles from {baci_path.name}...")
baci_dtypes = {'t': np.int16, 'k': np.int32, 'i': np.int16, 'j': np.int16, 'v': np.float32}

chunk_size = 250000 
with pd.read_csv(baci_path, dtype=baci_dtypes, chunksize=chunk_size) as reader:
    for chunk in tqdm(reader, desc="Processing Trade Blocks"):
        chunk['exp_iso'] = chunk['i'].map(baci_num_to_iso3)
        chunk['imp_iso'] = chunk['j'].map(baci_num_to_iso3)
        
        chunk = chunk.dropna(subset=['exp_iso', 'imp_iso'])
        chunk = chunk[chunk['exp_iso'].isin(iso3_to_graph_indices) & chunk['imp_iso'].isin(iso3_to_graph_indices)]
        
        if chunk.empty:
            continue
            
        chunk['k_str'] = chunk['k'].astype(str).str.zfill(6)
        
        for ch_name, prefixes in ECOLOGICAL_TRADE_MAPPING.items():
            ch_idx = channel_idx_map[ch_name]
            mask = chunk['k_str'].str.startswith(prefixes)
            sub_df = chunk[mask]
            
            if sub_df.empty:
                continue
                
            agg_flow = sub_df.groupby(['exp_iso', 'imp_iso'])['v'].sum().reset_index()
            
            for row in agg_flow.itertuples():
                exp_idxs = iso3_to_graph_indices[row.exp_iso]
                imp_idxs = iso3_to_graph_indices[row.imp_iso]
                trade_value = float(row.v)
                
                for e_idx in exp_idxs:
                    for i_idx in imp_idxs:
                        raw_trade_matrices[ch_idx, e_idx, i_idx] += trade_value

# Perform log-scale normalization to smooth out economic skews
normalized_trade_matrices = np.log1p(raw_trade_matrices)

# Convert to a clean PyTorch tensor and save to disk
trade_channels_tensor = torch.tensor(normalized_trade_matrices, dtype=torch.float32)

print(f"\n💾 Saving structured tensor to {output_path}...")
torch.save({
    'trade_tensor': trade_channels_tensor,
    'channels_order': channel_keys,
    'locations_order': all_country_names
}, output_path)

print("✅ Step 2b Processing Complete! File is ready for the main script.")

📦 Compiling multi-channel trade profiles from BACI_HS22_Y2024_V202601.csv...


Processing Trade Blocks: 0it [00:00, ?it/s]


💾 Saving structured tensor to c:\Users\simon\Documents\GitHub\horizon-scanner\data\trade_channels.pt...
✅ Step 2b Processing Complete! File is ready for the main script.
